# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/menna890/-Explaining-Search-Performance-Gaps-Using-Ranking-Signals/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked Actions + Reason Codes

### What the queue does
The refresh queue ranks every page by a **final score (0–100)** that blends:
- **70%** machine-learning model probability (catches complex patterns)
- **30%** rule-based baseline (transparent, no overfit)

### How to read the queue
| Priority | Action | When | Why |
|----------|--------|------|-----|
| 1 | `expand_and_refresh` | Thin content + visible | Low word count hurts ranking potential |
| 2 | `refresh_and_review_ctr` | Low CTR + visible | Snippet or title isn't compelling |
| 3 | `refresh_and_review_engagement` | Low engagement + visible | Content doesn't satisfy searchers |
| 4 | `refresh` | Old / never optimized / declining | Time decay or neglect |
| 5 | `monitor` | No strong signals | Check again next quarter |

### Reason codes humans can trust
Every row carries a `+`-separated reason code:
- `old` — older than median content age
- `never_optimized` — zero optimization history
- `thin` — word count below median
- `no_keyword` — missing keyword metadata
- `model_decline_risk` — model probability ≥ 65%
- `visible_model_opportunity` — model sees potential + impressions ≥ 500
 
### Measured Performance (Verified via Report)
- **High-confidence rows** (16,806 rows): **81.6%** hit rate (0.820 Precision@50 for CatBoost)
- **Medium-confidence rows** (42,240 rows): **55.4%** hit rate
- **Baseline-only rules**: **40.0%** hit rate — better than random, but well below ML performance
- **Overall Base Rate**: **38.5%** of pages in queue are truly below peer-median CTR
- The model is **directional** — it ranks probability, it doesn't predict certainty

In [6]:
# Load the queue and show ranked actions with reasons
import pandas as pd
from pathlib import Path

queue_path = Path("../outputs/refresh_queue.csv")
df = pd.read_csv(queue_path)

print(f"Total rows in queue: {len(df):,}")
print(f"\n{'='*70}")
print("TOP 20 HIGHEST-PRIORITY PAGES")
print(f"{'='*70}")

top20 = df.head(20)[['final_rank', 'content_hash_id', 'final_refresh_score', 
                      'confidence', 'suggested_action', 'final_reason_codes',
                      'total_impressions', 'ga4_sessions', 'is_below_peer_median']]

print(top20.to_string(index=False))

print(f"\n{'='*70}")
print("ACTION DISTRIBUTION")
print(f"{'='*70}")
print(df['suggested_action'].value_counts().to_string())

print(f"\n{'='*70}")
print("CONFIDENCE DISTRIBUTION")
print(f"{'='*70}")
print(df['confidence'].value_counts().to_string())

print(f"\n{'='*70}")
print("REASON CODE FREQUENCY (top 10)")
print(f"{'='*70}")
reason_counts = {}
for reasons in df['final_reason_codes']:
    for r in str(reasons).split('+'):
        reason_counts[r] = reason_counts.get(r, 0) + 1
for reason, count in sorted(reason_counts.items(), key=lambda x: -x[1])[:10]:
    print(f"  {reason:<30} {count:>6,}")

print(f"\n{'='*70}")
print("PRECISION BY CONFIDENCE LEVEL")
print(f"{'='*70}")
for conf in ['high', 'medium', 'low']:
    subset = df[df['confidence'] == conf]
    if len(subset) > 0:
        print(f"  {conf:<10} {len(subset):>6,} rows  "
              f"hit rate: {subset['is_below_peer_median'].mean():.3f}")

Total rows in queue: 118,092

TOP 20 HIGHEST-PRIORITY PAGES
 final_rank          content_hash_id  final_refresh_score confidence   suggested_action                                                                                    final_reason_codes  total_impressions  ga4_sessions  is_below_peer_median
          1 content_ac59cb40a7518023            90.980332       high expand_and_refresh                                                old+never_optimized+thin+no_keyword+model_decline_risk              102.0           0.0                     1
          2 content_a9ebfde31be778c4            90.592591       high            refresh                                                     old+never_optimized+no_keyword+model_decline_risk              101.0           0.0                     1
          3 content_1df31d5ae96d8fe5            90.333242       high            refresh                                                     old+never_optimized+no_keyword+model_decline_risk              15

## 2. Intended Use and Limits

### Who uses this
| Role | What they do with the queue |
|------|----------------------------|
| **Content editors** | Review high-confidence `refresh` rows first — check if the page still matches search intent |
| **SEO analysts** | Use `refresh_and_review_ctr` to identify snippet/title opportunities |
| **Content strategists** | Use `expand_and_refresh` to find thin pages with traffic potential |

### What this is (decision-support)
- A **ranked suggestion list** — not an automatic publish button
- Built on **observed patterns** in 90 days of anonymized data
- The model learns "what tended to be below median CTR" — not "what will happen"

### Where it stops being valid
| Scenario | Why it breaks |
|----------|---------------|
| **New product launch** | No historical data, patterns don't apply |
| **Algorithm update** | Google's ranking signals changed — model needs retrain |
| **Seasonal content** | Holiday pages behave differently in off-season |
| **Major site redesign** | URL structure, internal links, or templates changed |
| **Small sample client** | &lt; 100 pages = noisy statistics, unreliable percentiles |

### Careful words we use
- **"Observed"** — what we saw in the data, not a universal law
- **"Directional"** — points you toward likely problems, not guaranteed fixes
- **"Decision-support"** — informs human judgment, doesn't replace it
- **"Measured"** — based on calculated metrics, not subjective quality

In [7]:
print(f"{'='*70}")
print("DATA LIMITS CHECK")
print(f"{'='*70}")

# Small clients
client_counts = df['client_hash_id'].value_counts()
small_clients = (client_counts < 100).sum()
print(f"Clients with < 100 pages: {small_clients} "
      f"({small_clients/len(client_counts)*100:.1f}%)")

# Age distribution
print(f"\nContent age distribution:")
print(df['content_age_days'].describe()[['min', '25%', '50%', '75%', 'max']]
      .to_string())

# Pages with very few impressions (near the 100 threshold)
low_vis = df[df['total_impressions'] < 200]
print(f"\nPages with < 200 impressions: {len(low_vis):,} "
      f"({len(low_vis)/len(df)*100:.1f}%) — near threshold, noisy")

# Check for missing critical fields
missing_ctr = df['ctr'].isna().sum()
print(f"\nMissing CTR: {missing_ctr:,} rows")

DATA LIMITS CHECK
Clients with < 100 pages: 18 (38.3%)

Content age distribution:
min      1.0
25%     74.0
50%    193.0
75%    266.0
max    494.0

Pages with < 200 impressions: 15,887 (13.5%) — near threshold, noisy

Missing CTR: 0 rows


## 3. Human Review + The No-Go List

### What a person MUST check before acting

| Check | Why | How |
|-------|-----|-----|
| **Does the page still match search intent?** | Intent shifts; "best laptops 2025" ≠ "best laptops 2024" | Read top 3 ranking pages, compare angle |
| **Is the decline real or seasonal?** | Q4 holiday pages drop in January | Compare year-over-year, not month-over-month |
| **Did a competitor publish something better?** | Relative quality matters | Manual SERP check for target keywords |
| **Is the technical setup broken?** | Core Web Vitals, mobile usability | PageSpeed Insights, Search Console |
| **Would refresh actually help?** | Some pages are dead ends | Check if page has backlinks, internal links, or strategic value |

### The no-go list (NEVER automate)

| Never | Because |
|-------|---------|
| **Auto-publish refreshed content** | Quality control, brand voice, legal review |
| **Delete pages based on score alone** | Low score ≠ no value (support docs, legal pages) |
| **Change URLs** | Breaks backlinks, bookmarks, internal links |
| **Redirect without checking** | Chain redirects, redirect loops, lost equity |
| **Apply same template to all "thin" pages** | Different intents need different depth |
| **Ignore client-specific context** | A "thin" product page may be exactly right |

### Red flags that override the model
- Page is **brand homepage** or **core product page** → manual review only
- Page has **manual action** in Search Console → fix penalty first
- Page is **canonical target** for others → don't change without SEO lead
- Client says **"don't touch this page"** → respect it, no exceptions

In [8]:
print(f"{'='*70}")
print("PAGES REQUIRING EXTRA HUMAN REVIEW")
print(f"{'='*70}")

# Very high impressions + low score = might be important but model disagrees
important_but_low = df[(df['total_impressions'] > 10000) & 
                       (df['final_refresh_score'] < 30)]
print(f"High-traffic pages with low score: {len(important_but_low):,}")
print("  → Could be: homepage, core product, or seasonal page")
print("  → Action: MANUAL REVIEW ONLY, never auto-refresh")

# Very old + high confidence = previous work didn't stick (using days_since_update as proxy)
old_and_tried = df[(df['content_age_days'] > 730) & 
                   (df['days_since_update'] > 365) &   # proxy: not updated in a year
                   (df['confidence'] == 'high')]
print(f"\nOld + not recently updated + high confidence: {len(old_and_tried):,}")
print("  → Previous optimization may have failed — why?")
print("  → Action: Deep audit before second attempt")

# Score = 4 baseline + model agrees = strongest signal
# Note: baseline_score is 0-4, we check for high values
strongest_signal = df[(df['baseline_score'] >= 3) & 
                      (df['best_model_probability'] > 0.7)]
print(f"\nMaximum signal (baseline>=3 + model>70%): {len(strongest_signal):,}")
print("  → Highest confidence in the entire queue")
print("  → Action: Prioritize for review, but still check manually")

print(f"\n{'='*70}")
print("NO-GO VERIFICATION")
print(f"{'='*70}")
print("Checking for potential no-go scenarios in data...")

# Any pages with zero impressions after filter? (shouldn't happen, but verify)
zero_imp = df[df['total_impressions'] == 0]
print(f"Pages with zero impressions: {len(zero_imp)} (expected: 0)")

PAGES REQUIRING EXTRA HUMAN REVIEW
High-traffic pages with low score: 6,309
  → Could be: homepage, core product, or seasonal page
  → Action: MANUAL REVIEW ONLY, never auto-refresh

Old + not recently updated + high confidence: 0
  → Previous optimization may have failed — why?
  → Action: Deep audit before second attempt

Maximum signal (baseline>=3 + model>70%): 2,921
  → Highest confidence in the entire queue
  → Action: Prioritize for review, but still check manually

NO-GO VERIFICATION
Checking for potential no-go scenarios in data...
Pages with zero impressions: 0 (expected: 0)


## 4. Monitoring / Retrain Triggers

### What tells you recommendations went stale

| Signal | Threshold | Action |
|--------|-----------|--------|
| **Precision@50 drops** | &lt; 0.55 for 2 consecutive weeks | Investigate data drift |
| **Base rate shifts** | `is_below_peer_median` rate changes &gt; ±10% | CTR patterns changed |
| **Feature importance flips** | Top 3 features change order | Model learning different patterns |
| **New client onboarding** | &gt; 20% of rows from unseen client | Check if patterns generalize |
| **Seasonal event** | Black Friday, tax season, etc. | Expect temporary distortion |

### Automated checks to implement

```python
# Weekly monitoring script (pseudocode)
if precision_at_50_this_week &lt; 0.55:
    alert("Model performance degraded — check for data drift")
    
if mean(content_age_days) &gt; mean_last_quarter * 1.2:
    alert("Content aging faster than usual — refresh cycle may need tuning")
    
if new_clients / total_clients &gt; 0.2:
    alert("High new-client ratio — validate model generalization")
```

### Baseline Model Status
- **Best Model Selected**: `CatBoost` (ROC AUC: 0.850, Precision@50: 0.820)
- **Validation Split**: GroupShuffleSplit by `client_hash_id`

In [9]:
import json
with open(Path("../outputs/model_results.json")) as f:
    results = json.load(f)

print(f"{'='*70}")
print("MONITORING BASELINE (current state)")
print(f"{'='*70}")

# Current performance
best_model = results['best_model']['name']
best_p50 = results['best_model']['metrics']['precision_at_50']
best_auc = results['best_model']['metrics']['roc_auc']

print(f"Best model: {best_model}")
print(f"Current P@50: {best_p50:.4f}")
print(f"Current AUC:  {best_auc:.4f}")

# Thresholds
P50_THRESHOLD = 0.55
AUC_THRESHOLD = 0.55

print(f"\n{'='*70}")
print("RETRAIN TRIGGER CHECK")
print(f"{'='*70}")

if best_p50 < P50_THRESHOLD:
    print(f"⚠️  TRIGGER: P@50 ({best_p50:.4f}) < threshold ({P50_THRESHOLD})")
else:
    print(f"✅ P@50 OK: {best_p50:.4f} >= {P50_THRESHOLD}")

if best_auc < AUC_THRESHOLD:
    print(f"⚠️  TRIGGER: AUC ({best_auc:.4f}) < threshold ({AUC_THRESHOLD})")
else:
    print(f"✅ AUC OK: {best_auc:.4f} >= {AUC_THRESHOLD}")

# Data freshness check
print(f"\n{'='*70}")
print("DATA FRESHNESS METRICS")
print(f"{'='*70}")

print(f"Content age (days):")
print(df['content_age_days'].describe()[['mean', '50%', 'max']].to_string())

print(f"\nDays since last update:")
print(df['days_since_update'].describe()[['mean', '50%', 'max']].to_string())

# Client distribution drift check
client_pct = df['client_hash_id'].value_counts(normalize=True).head(5)
print(f"\nTop 5 clients by row %:")
print(client_pct.to_string())

# Simulate: if any client > 30%, flag concentration risk
max_client_pct = client_pct.iloc[0]
if max_client_pct > 0.30:
    print(f"\n⚠️  CONCENTRATION RISK: Top client = {max_client_pct:.1%} of data")
else:
    print(f"\n✅ Client distribution OK (max = {max_client_pct:.1%})")

MONITORING BASELINE (current state)
Best model: CatBoost
Current P@50: 0.8200
Current AUC:  0.8505

RETRAIN TRIGGER CHECK
✅ P@50 OK: 0.8200 >= 0.55
✅ AUC OK: 0.8505 >= 0.55

DATA FRESHNESS METRICS
Content age (days):
mean    194.125428
50%     193.000000
max     494.000000

Days since last update:
mean      8.528368
50%       0.000000
max     303.000000

Top 5 clients by row %:
client_hash_id
client_73cda7b4e4f265ea    0.197092
client_62f4a7e64f5e0096    0.166015
client_23a62021009f63c4    0.111125
client_08a6a72ff48e62c0    0.080065
client_e547b89c05043229    0.074645

✅ Client distribution OK (max = 19.7%)


## 5. Exports for the Paper

### Files this notebook produces

All outputs land in `work/outputs/` — the paper builds on these:

| File | What it is | Used in paper section |
|------|-----------|----------------------|
| `refresh_queue.csv` | Ranked queue with scores, actions, reasons | Results, Appendix A |
| `model_results.json` | Model comparison metrics | Methods, Results |
| `summary.json` | Pipeline metadata | Methods |
| `charts/*.svg` | 5 visualizations | Figures 2–6 |

### Key figures to reuse

| Figure | File | Caption suggestion |
|--------|------|-------------------|
| Model comparison | `model_comparison.svg` | "Precision@50 comparison across five classifiers and baseline rules" |
| Feature importance | `top_feature_importance.svg` | "Top 12 features by importance (CatBoost)" |
| Action distribution | `action_mix.svg` | "Recommended action distribution in final queue" |
| Confidence levels | `confidence_mix.svg` | "Queue confidence segmentation" |
| Reason codes | `top_reason_codes.svg` | "Most frequent reason codes" |

### Reproducibility checklist

- [x] All code in version-controlled repository
- [x] Random seed fixed (42) for all stochastic operations
- [x] GroupShuffleSplit prevents client leakage
- [x] Target computed from train-only median CTR
- [x] No URLs, titles, client names, or private identifiers in outputs
- [x] All claims use careful language (observed, directional, decision-support)

In [10]:
# Export verification + copy for paper
import pandas as pd
import json  # ← NEW
from pathlib import Path
import shutil

OUTPUT_DIR = Path("../outputs")
PAPER_DIR = OUTPUT_DIR / "for_paper"
PAPER_DIR.mkdir(exist_ok=True)

print(f"{'='*70}")
print("EXPORT VERIFICATION")
print(f"{'='*70}")

# Verify all expected files exist
expected_files = [
    "refresh_queue.csv",
    "model_results.json", 
    "summary.json",
    "charts/action_mix.svg",
    "charts/confidence_mix.svg",
    "charts/top_reason_codes.svg",
    "charts/top_feature_importance.svg",
    "charts/model_comparison.svg",
]

missing = []
for f in expected_files:
    path = OUTPUT_DIR / f
    if path.exists():
        size = path.stat().st_size
        print(f"  ✅ {f:<40} {size:>10,} bytes")
    else:
        print(f"  ❌ {f:<40} MISSING")
        missing.append(f)

if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

# Copy to paper directory
print(f"\n{'='*70}")
print("COPYING TO PAPER DIRECTORY")
print(f"{'='*70}")

for f in expected_files:
    src = OUTPUT_DIR / f
    dst = PAPER_DIR / Path(f).name
    if src.is_file():
        shutil.copy2(src, dst)
        print(f"  📄 {f} → {dst}")
    else:
        # Copy directory contents for charts
        dst_dir = PAPER_DIR / "charts"
        dst_dir.mkdir(exist_ok=True)
        for chart in src.glob("*"):
            shutil.copy2(chart, dst_dir / chart.name)
            print(f"  📊 {chart.name} → {dst_dir / chart.name}")
        break

# Generate paper-ready summary stats
print(f"\n{'='*70}")
print("PAPER-READY SUMMARY STATS")
print(f"{'='*70}")

df = pd.read_csv(OUTPUT_DIR / "refresh_queue.csv")

# FIX: Use json module instead of pd.read_json
with open(OUTPUT_DIR / "model_results.json") as f:
    results = json.load(f)

stats = {
    "n_pages_scored": int(len(df)),
    "n_high_confidence": int((df['confidence'] == 'high').sum()),
    "n_medium_confidence": int((df['confidence'] == 'medium').sum()),
    "n_low_confidence": int((df['confidence'] == 'low').sum()),
    "best_model": results['best_model']['name'],
    "best_model_p50": float(results['best_model']['metrics']['precision_at_50']),
    "best_model_auc": float(results['best_model']['metrics']['roc_auc']),
    "top_feature_1": results['best_model']['top_features'][0]['feature'],
    "top_feature_1_importance": float(results['best_model']['top_features'][0]['importance']),
    "action_refresh": int((df['suggested_action'] == 'refresh').sum()),
    "action_monitor": int((df['suggested_action'] == 'monitor').sum()),
    "action_expand": int((df['suggested_action'] == 'expand_and_refresh').sum()),
}

with open(PAPER_DIR / "paper_stats.json", "w") as f:
    json.dump(stats, f, indent=2)

print(json.dumps(stats, indent=2))

print(f"\n{'='*70}")
print("ALL EXPORTS READY FOR PAPER")
print(f"{'='*70}")
print(f"Location: {PAPER_DIR}")

EXPORT VERIFICATION
  ✅ refresh_queue.csv                        29,444,174 bytes
  ✅ model_results.json                            6,574 bytes
  ✅ summary.json                                  1,055 bytes
  ✅ charts/action_mix.svg                         1,696 bytes
  ✅ charts/confidence_mix.svg                     1,100 bytes
  ✅ charts/top_reason_codes.svg                   2,838 bytes
  ✅ charts/top_feature_importance.svg             3,720 bytes
  ✅ charts/model_comparison.svg                   1,661 bytes

COPYING TO PAPER DIRECTORY
  📄 refresh_queue.csv → ..\outputs\for_paper\refresh_queue.csv
  📄 model_results.json → ..\outputs\for_paper\model_results.json
  📄 summary.json → ..\outputs\for_paper\summary.json
  📄 charts/action_mix.svg → ..\outputs\for_paper\action_mix.svg
  📄 charts/confidence_mix.svg → ..\outputs\for_paper\confidence_mix.svg
  📄 charts/top_reason_codes.svg → ..\outputs\for_paper\top_reason_codes.svg
  📄 charts/top_feature_importance.svg → ..\outputs\for_paper\to